# NILM Disaggregation — Architecture Comparison (4 Models)

## Experiment Design
Compare four architectures on the **same data, same split, same loss, same metrics** for fair evaluation.

| # | Model | Key Feature | Parameters |
|---|-------|-------------|------------|
| 1 | **Seq2Point CNN** | Classic 5-layer CNN (Zhang 2018) | ~1.5M |
| 2 | **ResWaveNet-Transformer** | Residual dilated CNN + Multi-Head Attention | ~1.2M |
| 3 | **CNN-BiGRU-Attention** | Lighter recurrent variant | ~1.8M |
| 4 | **MLP Baseline** | No temporal modelling (control) | ~1.0M |

**Training:** Per-appliance models (1 model per appliance per architecture = 8 total)  
**Split:** Chronological 70/15/15  
**Loss:** Huber (δ=1.0) — robust regression  
**Goal:** Determine which architecture best tracks Motor's noisy oscillations

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv1D, Dense, Flatten, Dropout, Input, Add,
    Bidirectional, LSTM, GRU,
    LayerNormalization, BatchNormalization, GaussianNoise,
    GlobalAveragePooling1D, MultiHeadAttention
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110

WINDOW   = 99
STRIDE   = 4
MIDPOINT = WINDOW // 2
APPLIANCES = ['AC', 'Motor']
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

---
## 1. Load Data & Chronological Split

In [ ]:
df = pd.read_pickle('data/iawe_preprocessed.pkl')

FEATURE_COLS = ['Aggregate', 'Vrms', 'Irms', 'Reactive_Q',
                'Apparent_Power', 'Power_Factor',
                'Delta_P', 'Rolling_Mean', 'Rolling_Std', 'Hour']
N_FEATURES = len(FEATURE_COLS)

# ── Z-Score Standardisation ──
feat_stats = {}
feat_matrix = np.zeros((len(df), N_FEATURES), dtype='float32')
for j, col in enumerate(FEATURE_COLS):
    mu, sig = df[col].values.mean(), df[col].values.std()
    if sig < 1e-6: sig = 1.0
    feat_stats[col] = {'mu': mu, 'sig': sig}
    feat_matrix[:, j] = (df[col].values - mu) / sig

app_stats = {}
for name in APPLIANCES:
    mu, sig = df[name].values.mean(), df[name].values.std()
    if sig < 1e-6: sig = 1.0
    app_stats[name] = {'mu': mu, 'sig': sig}

# ── Sliding windows ──
num_windows = (len(feat_matrix) - WINDOW) // STRIDE
X_all = np.zeros((num_windows, WINDOW, N_FEATURES), dtype='float32')
Y_all = {name: np.zeros(num_windows, dtype='float32') for name in APPLIANCES}

for i in range(num_windows):
    s = i * STRIDE
    X_all[i] = feat_matrix[s : s + WINDOW]
    for name in APPLIANCES:
        val = df[name].values[s + MIDPOINT]
        Y_all[name][i] = (val - app_stats[name]['mu']) / app_stats[name]['sig']

# ── CHRONOLOGICAL SPLIT ──
n_train = int(num_windows * 0.70)
n_val   = int(num_windows * 0.15)

X_train, X_val, X_test = X_all[:n_train], X_all[n_train:n_train+n_val], X_all[n_train+n_val:]
Y_split = {}
for name in APPLIANCES:
    Y_split[name] = {
        'train': Y_all[name][:n_train],
        'val':   Y_all[name][n_train:n_train+n_val],
        'test':  Y_all[name][n_train+n_val:],
    }

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')

---
## 2. Model Architectures

In [ ]:
def build_seq2point():
    """Model 1: Classic Seq2Point CNN (Zhang 2018) with BatchNorm"""
    inp = Input(shape=(WINDOW, N_FEATURES))
    x = GaussianNoise(0.03)(inp)
    x = Conv1D(30, 10, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(30,  8, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(40,  6, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(50,  5, activation='relu')(x)
    x = Conv1D(50,  5, activation='relu')(x)
    x = Flatten()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='linear')(x)
    model = Model(inputs=inp, outputs=out, name='Seq2Point_CNN')
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

def build_reswavenet_transformer():
    """Model 2: ResWaveNet-Transformer (our best)"""
    inp = Input(shape=(WINDOW, N_FEATURES))
    x = GaussianNoise(0.03)(inp)

    # Residual Dilated Block 1
    res = Conv1D(64, 1, padding='same')(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=1, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=2, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=4, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=8, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Add()([x, res])
    x = LayerNormalization()(x)

    # Residual Dilated Block 2
    res2 = x
    x = Conv1D(64, 3, padding='same', dilation_rate=16, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=32, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Add()([x, res2])
    x = LayerNormalization()(x)

    # Transformer block
    attn = MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(x, x)
    x = Add()([x, attn])
    x = LayerNormalization()(x)
    ff = Dense(256, activation='relu')(x)
    ff = Dropout(0.2)(ff)
    ff = Dense(64)(ff)
    x = Add()([x, ff])
    x = LayerNormalization()(x)

    x = GlobalAveragePooling1D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    out = Dense(1, activation='linear')(x)

    model = Model(inputs=inp, outputs=out, name='ResWaveNet_Transformer')
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

def build_cnn_bigru_attn():
    """Model 3: CNN-BiGRU-Attention (lighter recurrent variant)"""
    inp = Input(shape=(WINDOW, N_FEATURES))
    x = GaussianNoise(0.03)(inp)

    x = Conv1D(64, 3, padding='same', dilation_rate=1, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=2, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', dilation_rate=4, activation='relu')(x)
    x = BatchNormalization()(x)

    x = Bidirectional(GRU(96, return_sequences=True, dropout=0.2))(x)

    attn = MultiHeadAttention(num_heads=4, key_dim=24, dropout=0.1)(x, x)
    x = Add()([x, attn])
    x = LayerNormalization()(x)

    x = GlobalAveragePooling1D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(1, activation='linear')(x)

    model = Model(inputs=inp, outputs=out, name='CNN_BiGRU_Attention')
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

def build_mlp():
    """Model 4: Deep MLP Baseline (no temporal modelling)"""
    inp = Input(shape=(WINDOW, N_FEATURES))
    x = Flatten()(inp)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    out = Dense(1, activation='linear')(x)
    model = Model(inputs=inp, outputs=out, name='MLP_Baseline')
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

print('All 4 architectures ready. Loss = Huber (δ=1.0)')

---
## 3. Train All Models

In [ ]:
callbacks_fn = lambda: [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, min_lr=1e-6, verbose=0),
]

BUILDERS = {
    'Seq2Point CNN':         build_seq2point,
    'ResWaveNet-Transformer': build_reswavenet_transformer,
    'CNN-BiGRU-Attention':   build_cnn_bigru_attn,
    'MLP Baseline':          build_mlp,
}

all_models = {}
all_histories = {}

for model_name, builder in BUILDERS.items():
    for appliance in APPLIANCES:
        key = (model_name, appliance)
        print(f'\n{"="*60}\n{model_name} → {appliance}\n{"="*60}')

        m = builder()
        h = m.fit(
            X_train, Y_split[appliance]['train'],
            validation_data=(X_val, Y_split[appliance]['val']),
            epochs=50, batch_size=128,
            callbacks=callbacks_fn(), verbose=1
        )
        all_models[key] = m
        all_histories[key] = h

print('\n[OK] All 8 models trained.')

---
## 4. Evaluation & Comparison

In [ ]:
def evaluate(model, X, Y_zscore, appliance, threshold_w=15):
    pred_z = model.predict(X, verbose=0).flatten()
    mu, sig = app_stats[appliance]['mu'], app_stats[appliance]['sig']

    pred_w = np.clip((pred_z * sig) + mu, 0, None)
    true_w = np.clip((Y_zscore * sig) + mu, 0, None)

    mae  = mean_absolute_error(true_w, pred_w)
    rmse = np.sqrt(mean_squared_error(true_w, pred_w))
    r2   = r2_score(true_w, pred_w) if np.std(true_w) > 0 else 0
    nde  = np.sum((pred_w - true_w)**2) / (np.sum(true_w**2) + 1e-6)
    da   = 1 - np.sum(np.abs(pred_w - true_w)) / (2 * np.sum(np.abs(true_w)) + 1e-6)
    sae  = abs(pred_w.sum() - true_w.sum()) / (true_w.sum() + 1e-6)

    true_on = (true_w > threshold_w).astype(int)
    pred_on = (pred_w > threshold_w).astype(int)
    acc = accuracy_score(true_on, pred_on)
    f1  = f1_score(true_on, pred_on, average='macro', zero_division=0)

    metrics = {'MAE(W)': mae, 'RMSE(W)': rmse, 'R²': r2, 'NDE': nde,
               'DA': da, 'Accuracy': acc, 'F1': f1, 'SAE': sae}
    return metrics, true_w, pred_w

rows = []
preds = {}

for (model_name, appliance), model in all_models.items():
    m, true_w, pred_w = evaluate(model, X_test, Y_split[appliance]['test'], appliance)
    rows.append({'Model': model_name, 'Appliance': appliance, **m})
    preds[(model_name, appliance)] = {'true': true_w, 'pred': pred_w}

df_results = pd.DataFrame(rows)

# ── Display formatted table ──
fmt = df_results.copy()
for col in ['MAE(W)', 'RMSE(W)']: fmt[col] = fmt[col].map('{:.1f}'.format)
for col in ['R²', 'NDE', 'F1', 'SAE']: fmt[col] = fmt[col].map('{:.4f}'.format)
for col in ['DA', 'Accuracy']: fmt[col] = fmt[col].map('{:.1%}'.format)

print('\n' + '='*80)
print('ARCHITECTURE COMPARISON — Full Metrics')
print('='*80)
display(fmt.set_index(['Model', 'Appliance']))

In [ ]:
# ── Waveform Comparison Plot ──
N = 600
ac_true = preds[('ResWaveNet-Transformer', 'AC')]['true']
best_start, best_score = 0, 0
for s in range(0, len(ac_true) - N, 50):
    score = np.sum(ac_true[s:s+N] > 50) + np.sum(preds[('ResWaveNet-Transformer', 'Motor')]['true'][s:s+N] > 15)
    if score > best_score: best_score, best_start = score, s
sl = slice(best_start, best_start + N)

fig, axes = plt.subplots(len(APPLIANCES), 1, figsize=(18, 5*len(APPLIANCES)))
model_colors = {
    'Seq2Point CNN': '#e74c3c',
    'ResWaveNet-Transformer': '#2ecc71',
    'CNN-BiGRU-Attention': '#3498db',
    'MLP Baseline': '#9b59b6'
}

for i, app in enumerate(APPLIANCES):
    true_w = preds[('ResWaveNet-Transformer', app)]['true'][sl]
    axes[i].fill_between(range(len(true_w)), true_w, alpha=0.2, color='dodgerblue', label='Ground Truth')
    axes[i].plot(true_w, color='dodgerblue', linewidth=2, alpha=0.8)

    for mn in BUILDERS:
        pred_w = preds[(mn, app)]['pred'][sl]
        mae = mean_absolute_error(true_w, pred_w)
        axes[i].plot(pred_w, color=model_colors[mn], linewidth=1.5, linestyle='--',
                     label=f'{mn} (MAE={mae:.0f}W)')

    axes[i].set_title(f'{app} — Architecture Comparison', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Power (W)')
    axes[i].legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Time Step')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary: Best model per appliance ──
print('\n' + '='*60)
print('BEST MODEL PER APPLIANCE (by MAE)')
print('='*60)
for app in APPLIANCES:
    app_df = df_results[df_results['Appliance'] == app]
    best = app_df.loc[app_df['MAE(W)'].idxmin()]
    print(f'  {app:8s}: {best["Model"]:28s} → MAE = {best["MAE(W)"]:.1f}W, R² = {best["R²"]:.4f}, F1 = {best["F1"]:.4f}')

print('\n' + '='*60)
print('BEST MODEL PER APPLIANCE (by R²)')
print('='*60)
for app in APPLIANCES:
    app_df = df_results[df_results['Appliance'] == app]
    best = app_df.loc[app_df['R²'].idxmax()]
    print(f'  {app:8s}: {best["Model"]:28s} → R² = {best["R²"]:.4f}, MAE = {best["MAE(W)"]:.1f}W')